In [4]:
%load_ext autoreload
%autoreload 2

import os
import time
import pandas as pd
import psutil
import sys

# Add the parent directory to Python's module search path
sys.path.append(os.path.abspath('..'))
from src.bpe import BPETokenizer

# Ensure directories exist
os.makedirs('../vocabs/', exist_ok=True)
os.makedirs('../reports/', exist_ok=True)

# Use a larger slice of the text
with open('../data/raw/wikitext_sample.txt', 'r', encoding='utf-8') as f:
    text = f.read()[:500000] # Adjust slice as needed for speed vs accuracy

test_text = "Natural language processing requires highly efficient subword tokenization to handle morphologically rich dialects."

# Assignment requires these exact 6 sizes
target_sizes = [8000] 
results = []

for size in target_sizes:
    print(f"\n--- Processing BPE for {size} Vocab ---")
    tokenizer = BPETokenizer()
    vocab_file = f'../vocabs/bpe_{size}.json'
    
    process = psutil.Process(os.getpid())
    mem_before = process.memory_info().rss / (1024 * 1024)
    start_time = time.time()
    
    # SMART CHECK: If already trained, load it instantly!
    if os.path.exists(vocab_file):
        print(f"✅ Found existing {size} vocab file. Loading instantly...")
        tokenizer.load(vocab_file)
        train_time = 0.0 
        mem_used = 0.0
    else:
        print(f"⏳ Training from scratch (This will take a few minutes)...")
        tokenizer.train(text, target_vocab_size=size)
        tokenizer.save(vocab_file)
        
        mem_after = process.memory_info().rss / (1024 * 1024)
        mem_used = max(0, mem_after - mem_before)
        train_time = time.time() - start_time
    
    # Evaluate Metrics for the Assignment Report
    eval_data = tokenizer.encode(test_text)
    compression_ratio = len(test_text) / len(eval_data['tokens']) if eval_data['tokens'] else 0
    
    results.append({
        "Vocab Size": size,
        "Time (s)": round(train_time, 2),
        "Memory (MB)": round(mem_used, 2),
        "Avg Tokens/Word": round(eval_data['tokens_per_word'], 3),
        "OOV Rate": round(eval_data['oov_rate'], 4),
        "Compression Ratio": round(compression_ratio, 2),
        "Example": " | ".join(eval_data['tokens'][:8]) + "..."
    })

# Display Results Table
df_results = pd.DataFrame(results)
display(df_results)

# Save for the analytical report
df_results.to_csv('../reports/performance_metrics.csv', index=False)
print("\nMetrics successfully saved to ../reports/performance_metrics.csv")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload

--- Processing BPE for 8000 Vocab ---
⏳ Training from scratch (This will take a few minutes)...
  -> Starting BPE Training for target size: 8000...
  -> Base character vocabulary size: 138
  -> Performing 7862 merges...
    ... Completed 1000 / 7862 merges
    ... Completed 2000 / 7862 merges
    ... Completed 3000 / 7862 merges
    ... Completed 4000 / 7862 merges
    ... Completed 5000 / 7862 merges
    ... Completed 6000 / 7862 merges
    ... Completed 7000 / 7862 merges
  -> Training complete in 125.76 seconds.


,Vocab Size,Time (s),Memory (MB),Avg Tokens/Word,OOV Rate,Compression Ratio,Example
0,8000,125.78,0,2.143,0.0,3.83,natural</w> | language</w> | pro | ces | sing<...



Metrics successfully saved to ../reports/performance_metrics.csv


In [5]:
# --- 16K BPE TRAINING CELL ---
import os, time, psutil
import pandas as pd
from src.bpe import BPETokenizer

size = 16000
print(f"\n--- Processing BPE for {size} Vocab ---")

tokenizer_16k = BPETokenizer()
vocab_file = f'../vocabs/bpe_{size}.json'
csv_path = '../reports/performance_metrics.csv'

process = psutil.Process(os.getpid())
mem_before = process.memory_info().rss / (1024 * 1024)
start_time = time.time()

# SMART CHECK: If already trained, load it instantly!
if os.path.exists(vocab_file):
    print(f"✅ Found existing {size} vocab file. Loading instantly...")
    tokenizer_16k.load(vocab_file)
    train_time = 0.0 
    mem_used = 0.0
else:
    print(f"⏳ Training from scratch (This will take a few minutes)...")
    tokenizer_16k.train(text, target_vocab_size=size)
    tokenizer_16k.save(vocab_file)
    
    mem_after = process.memory_info().rss / (1024 * 1024)
    mem_used = max(0, mem_after - mem_before)
    train_time = time.time() - start_time

# Evaluate Metrics
eval_data = tokenizer_16k.encode(test_text)
compression_ratio = len(test_text) / len(eval_data['tokens']) if eval_data['tokens'] else 0

new_result = {
    "Vocab Size": size,
    "Time (s)": round(train_time, 2),
    "Memory (MB)": round(mem_used, 2),
    "Avg Tokens/Word": round(eval_data['tokens_per_word'], 3),
    "OOV Rate": round(eval_data['oov_rate'], 4),
    "Compression Ratio": round(compression_ratio, 2),
    "Example": " | ".join(eval_data['tokens'][:8]) + "..."
}

# Safely append to the existing CSV report
if os.path.exists(csv_path):
    df_existing = pd.read_csv(csv_path)
    # Check if 16K is already in there to avoid duplicate rows
    if size not in df_existing['Vocab Size'].values:
        df_updated = pd.concat([df_existing, pd.DataFrame([new_result])], ignore_index=True)
    else:
        df_updated = df_existing
        print(f"⚠️ {size} vocab metrics already exist in the report. Skipping append.")
else:
    df_updated = pd.DataFrame([new_result])

df_updated.to_csv(csv_path, index=False)
display(df_updated)


--- Processing BPE for 16000 Vocab ---
⏳ Training from scratch (This will take a few minutes)...
  -> Starting BPE Training for target size: 16000...
  -> Base character vocabulary size: 138
  -> Performing 15862 merges...
    ... Completed 1000 / 15862 merges
    ... Completed 2000 / 15862 merges
    ... Completed 3000 / 15862 merges
    ... Completed 4000 / 15862 merges
    ... Completed 5000 / 15862 merges
    ... Completed 6000 / 15862 merges
    ... Completed 7000 / 15862 merges
    ... Completed 8000 / 15862 merges
    ... Completed 9000 / 15862 merges
    ... Completed 10000 / 15862 merges
    ... Completed 11000 / 15862 merges
    ... Completed 12000 / 15862 merges
    ... Completed 13000 / 15862 merges
    ... Completed 14000 / 15862 merges
    ... Completed 15000 / 15862 merges
  -> Training complete in 199.97 seconds.


,Vocab Size,Time (s),Memory (MB),Avg Tokens/Word,OOV Rate,Compression Ratio,Example
0,8000,125.78,0,2.143,0.0,3.83,natural</w> | language</w> | pro | ces | sing<...
1,16000,200.01,0,1.786,0.0,4.60,natural</w> | language</w> | pro | ces | sing<...


In [8]:
# --- 32K BPE TRAINING CELL (INCREASED CAPACITY) ---
import os, time, psutil
import pandas as pd
from src.bpe import BPETokenizer

size = 32000
print(f"\n--- Processing BPE for {size} Vocab ---")

# 1. INCREASED READING CAPACITY: Read 2 Million characters instead of 500k
with open('../data/raw/wikitext_sample.txt', 'r', encoding='utf-8') as f:
    text_32k = f.read()[:2000000] 
print(f"Loaded {len(text_32k)} characters for training.")

tokenizer_32k = BPETokenizer()
vocab_file = f'../vocabs/bpe_{size}.json'
csv_path = '../reports/performance_metrics.csv'
test_text = "Natural language processing requires highly efficient subword tokenization to handle morphologically rich dialects."

process = psutil.Process(os.getpid())
mem_before = process.memory_info().rss / (1024 * 1024)
start_time = time.time()

# SMART CHECK: If already trained, load it instantly!
if os.path.exists(vocab_file):
    print(f"✅ Found existing {size} vocab file. Loading instantly...")
    tokenizer_32k.load(vocab_file)
    train_time = 0.0 
    mem_used = 0.0
else:
    print(f"⏳ Training from scratch (This will take several minutes due to increased capacity)...")
    # Make sure we pass the new text_32k variable here!
    tokenizer_32k.train(text_32k, target_vocab_size=size)
    tokenizer_32k.save(vocab_file)
    
    mem_after = process.memory_info().rss / (1024 * 1024)
    mem_used = max(0, mem_after - mem_before)
    train_time = time.time() - start_time

# Evaluate Metrics
eval_data = tokenizer_32k.encode(test_text)
compression_ratio = len(test_text) / len(eval_data['tokens']) if eval_data['tokens'] else 0

new_result = {
    "Vocab Size": size,
    "Time (s)": round(train_time, 2),
    "Memory (MB)": round(mem_used, 2),
    "Avg Tokens/Word": round(eval_data['tokens_per_word'], 3),
    "OOV Rate": round(eval_data['oov_rate'], 4),
    "Compression Ratio": round(compression_ratio, 2),
    "Example": " | ".join(eval_data['tokens'][:8]) + "..."
}

# Safely append to the existing CSV report
if os.path.exists(csv_path):
    df_existing = pd.read_csv(csv_path)
    if size not in df_existing['Vocab Size'].values:
        df_updated = pd.concat([df_existing, pd.DataFrame([new_result])], ignore_index=True)
    else:
        df_updated = df_existing
        print(f"⚠️ {size} vocab metrics already exist in the report. Skipping append.")
else:
    df_updated = pd.DataFrame([new_result])

df_updated.to_csv(csv_path, index=False)
display(df_updated)


--- Processing BPE for 32000 Vocab ---
Loaded 2000000 characters for training.
⏳ Training from scratch (This will take several minutes due to increased capacity)...
  -> Starting BPE Training for target size: 32000...
  -> Base character vocabulary size: 346
  -> Performing 31654 merges...
    ... Completed 1000 / 31654 merges
    ... Completed 2000 / 31654 merges
    ... Completed 3000 / 31654 merges
    ... Completed 4000 / 31654 merges
    ... Completed 5000 / 31654 merges
    ... Completed 6000 / 31654 merges
    ... Completed 7000 / 31654 merges
    ... Completed 8000 / 31654 merges
    ... Completed 9000 / 31654 merges
    ... Completed 10000 / 31654 merges
    ... Completed 11000 / 31654 merges
    ... Completed 12000 / 31654 merges
    ... Completed 13000 / 31654 merges
    ... Completed 14000 / 31654 merges
    ... Completed 15000 / 31654 merges
    ... Completed 16000 / 31654 merges
    ... Completed 17000 / 31654 merges
    ... Completed 18000 / 31654 merges
    ... Complet

,Vocab Size,Time (s),Memory (MB),Avg Tokens/Word,OOV Rate,Compression Ratio,Example
0,8000,125.78,0.00,2.143,0.0,3.83,natural</w> | language</w> | pro | ces | sing<...
1,16000,200.01,0.00,1.786,0.0,4.60,natural</w> | language</w> | pro | ces | sing<...
2,32000,238.56,2.33,1.786,0.0,4.60,natural</w> | language</w> | pro | ces | sing<...
